<a href="https://colab.research.google.com/github/khadws/Komputasi-Numerik/blob/main/SPL_3_Variabel(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Penyelesaian Sistem Persamaan Linear (SPL) 3 Variabel

## Latar belakang masalah
SPL muncul saat beberapa besaran tak diketahui saling terkait, misalnya arus pada rangkaian listrik, komposisi campuran, atau kesetimbangan gaya. Untuk 3 variabel, bentuknya:
```
a11*x + a12*y + a13*z = b1
a21*x + a22*y + a23*z = b2
a31*x + a32*y + a33*z = b3
```
Dalam bentuk matriks: **A · X = b**, dengan A matriks koefisien 3x3, X = [x, y, z] yang dicari, dan b vektor konstanta.

## Metode yang dipakai
| Metode | Peran di notebook | Ide dasar |
|---|---|---|
| Eliminasi Gauss + pivoting parsial | Metode utama | Ubah A menjadi segitiga atas, lalu hitung dari bawah ke atas |
| Aturan Cramer | Pembanding | Gunakan rasio determinan |
| `numpy.linalg.solve` | Validasi | Solver standar (dekomposisi LU) |

## Syarat solusi tunggal
SPL punya satu solusi hanya jika **det(A) ≠ 0**. Jika det(A) = 0, SPL tidak punya solusi atau punya tak hingga solusi. Program akan menampilkan pesan error pada kasus ini.

## Cara menjalankan
Jalankan cell dari atas ke bawah (Shift+Enter). Cell bawah memakai variabel dari cell atas, jadi jangan melompati urutan.

## Cell 1 - Import library
**Tujuan:** memuat NumPy.

**Penjelasan:** NumPy menyediakan array bertipe numerik, operasi vektor/matriks (`np.dot`, `@`), dan fungsi aljabar linear (`np.linalg`). Alias `np` adalah konvensi umum.

**Masalah umum:** jika muncul `ModuleNotFoundError`, jalankan `!pip install numpy`. Di Colab, NumPy sudah terpasang sehingga biasanya tidak ada masalah.

In [ ]:
import numpy as np   # library untuk array/matriks dan fungsi aljabar linear

## Cell 2 - Input data SPL
**Tujuan:** menyimpan SPL ke dalam variabel `A` (koefisien) dan `b` (konstanta).

**Cara membaca contoh:**
```
 2x +  y -  z =   8   -> baris 1 A = [ 2,  1, -1], b1 =   8
-3x -  y + 2z = -11   -> baris 2 A = [-3, -1,  2], b2 = -11
-2x +  y + 2z =  -3   -> baris 3 A = [-2,  1,  2], b3 =  -3
```
**Poin penting:**
- Tulis angka dengan titik desimal (`2.`) agar bertipe *float*. Jika semua integer, hasil pembagian bisa terpotong pada beberapa operasi.
- Koefisien yang tidak muncul harus ditulis **0**. Contoh: `2x + 3z = 5` menjadi `[2, 0, 3]`.
- Urutan kolom harus konsisten: kolom 1 = x, kolom 2 = y, kolom 3 = z.

**Masalah umum:** `A` bukan 3x3 atau panjang `b` tidak sama dengan jumlah baris `A` akan menyebabkan error dimensi.

In [ ]:
# Contoh SPL:
#  2x +  y -  z =   8
# -3x -  y + 2z = -11
# -2x +  y + 2z =  -3
A = np.array([[ 2.,  1., -1.],
              [-3., -1.,  2.],
              [-2.,  1.,  2.]])   # matriks koefisien, tipe float agar pembagian akurat
b = np.array([8., -11., -3.])     # vektor ruas kanan

n = len(b)                        # jumlah persamaan/variabel (=3)
print("Matriks A:\n", A)
print("Vektor b :", b)

## Cell 3 - Fungsi Eliminasi Gauss
**Tujuan:** menyelesaikan SPL dengan mengubah matriks menjadi bentuk segitiga atas.

**Tahapan algoritma:**
1. **Matriks augmented `[A|b]`.** `b` ditempel sebagai kolom ke-4 agar operasi baris berlaku untuk koefisien dan konstanta sekaligus. `np.hstack` menggabungkan kolom, dan `reshape(-1,1)` mengubah `b` menjadi kolom.
2. **Pivoting parsial.** Sebelum mengeliminasi kolom k, dipilih baris (dari k ke bawah) dengan nilai mutlak terbesar di kolom itu (`np.argmax`), lalu ditukar ke posisi pivot.
   - *Mengapa perlu?* Pivot bernilai 0 membuat pembagian gagal, dan pivot sangat kecil memperbesar galat pembulatan.
3. **Eliminasi maju.** Untuk tiap baris di bawah pivot: `faktor = elemen / pivot`, lalu `R_i = R_i - faktor * R_k`. Hasilnya, elemen di bawah diagonal menjadi 0.
4. **Pemeriksaan singular.** Jika pivot ≈ 0 (< 1e-12), SPL tidak punya solusi tunggal dan program berhenti dengan pesan error. Toleransi dipakai karena komputer jarang menghasilkan 0 yang persis.
5. **Substitusi mundur.** Dari baris terakhir: `z = b3'/a33'`. Lalu `y` memakai z yang sudah diketahui, lalu `x`, dengan rumus `x_i = (b_i - Σ a_ij·x_j) / a_ii`.

**Parameter `tampil`:** `True` mencetak tiap tahap supaya proses terlihat; `False` hanya mengembalikan hasil.

**Kompleksitas:** sekitar O(n³), sangat ringan untuk n = 3.

In [ ]:
def gauss_elimination(A, b, tampil=True):
    n = len(b)
    # 1. Matriks augmented [A | b] (copy agar data asli tidak berubah)
    M = np.hstack([A.astype(float), b.reshape(-1, 1)])

    # 2. Eliminasi maju
    for k in range(n - 1):
        # Pivoting parsial: cari baris dengan |nilai| terbesar di kolom k
        p = np.argmax(np.abs(M[k:, k])) + k
        if abs(M[p, k]) < 1e-12:
            raise ValueError("Matriks singular: tidak ada solusi tunggal.")
        if p != k:
            M[[k, p]] = M[[p, k]]          # tukar baris k dan p
            if tampil: print(f"Tukar baris {k+1} <-> {p+1}")
        # Nolkan elemen di bawah pivot
        for i in range(k + 1, n):
            faktor = M[i, k] / M[k, k]     # pengali eliminasi
            M[i, k:] -= faktor * M[k, k:]  # R_i = R_i - faktor * R_k
        if tampil:
            print(f"\nSetelah eliminasi kolom {k+1}:\n", np.round(M, 4))

    if abs(M[n-1, n-1]) < 1e-12:
        raise ValueError("Matriks singular: tidak ada solusi tunggal.")

    # 3. Substitusi mundur
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (M[i, -1] - np.dot(M[i, i+1:n], x[i+1:])) / M[i, i]
    return x

## Cell 4 - Jalankan solusi
**Tujuan:** memanggil fungsi dan menampilkan x, y, z.

**Cara membaca output tahap eliminasi (untuk contoh):**
- `Tukar baris 1 <-> 2`: kolom 1 punya nilai mutlak terbesar (|-3| = 3) di baris 2, sehingga baris 1 dan 2 ditukar.
- Matriks setelah "eliminasi kolom 1": kolom pertama di bawah pivot sudah 0.
- Matriks setelah "eliminasi kolom 2": seluruh elemen di bawah diagonal sudah 0 (bentuk segitiga atas).

**Hasil akhir yang diharapkan:** x = 2, y = 3, z = -1.

**Masalah umum:** muncul `Matriks singular`. Periksa apakah dua persamaan Anda sebenarnya kelipatan satu sama lain (misalnya persamaan 2 = 2 × persamaan 1), atau apakah ada kesalahan ketik pada koefisien.

In [ ]:
x = gauss_elimination(A, b)
print("\n=== HASIL ===")
for nama, nilai in zip(["x", "y", "z"], x):
    print(f"{nama} = {nilai:.6f}")

## Cell 5 - Verifikasi hasil
**Tujuan:** memastikan hasil benar, bukan sekadar "keluar angka".

**Tiga cara verifikasi:**
1. **Substitusi balik.** `A @ x` harus menghasilkan `b`. Operator `@` adalah perkalian matriks.
2. **Residual.** Selisih terbesar antara `A @ x` dan `b`. Nilainya harus sangat kecil (sekitar 1e-15), bukan persis 0, karena representasi bilangan desimal di komputer terbatas (galat *floating point*).
3. **Perbandingan dengan NumPy.** `np.allclose` menganggap dua array sama jika selisihnya di bawah toleransi kecil.

**Interpretasi:** jika `Sama dengan NumPy? True` dan residual ≈ 1e-15, implementasi Gauss Anda benar. Jika `False`, periksa ulang input atau pengali eliminasi.

In [ ]:
print("A @ x          :", A @ x)
print("b               :", b)
print("Residual (maks) :", np.max(np.abs(A @ x - b)))
print("numpy.linalg    :", np.linalg.solve(A, b))
print("Sama dengan NumPy?", np.allclose(x, np.linalg.solve(A, b)))

## Cell 6 - Pembanding: Aturan Cramer
**Tujuan:** menyelesaikan SPL dengan metode kedua sebagai pembanding.

**Rumus:** `x_i = det(A_i) / det(A)`, dengan `A_i` adalah matriks A yang kolom ke-i-nya diganti vektor `b`.
- `A.copy()` dipakai agar `A` asli tidak ikut berubah.
- `Ai[:, i] = b` mengganti seluruh kolom ke-i.

**Syarat:** `det(A) ≠ 0`. Jika 0, fungsi mengeluarkan error.

**Perbandingan dengan Gauss:**
| Aspek | Gauss | Cramer |
|---|---|---|
| Efisiensi | Cepat, O(n³) | Sangat lambat untuk n besar |
| Kestabilan numerik | Baik dengan pivoting | Kurang baik |
| Cocok untuk | Semua ukuran | Hanya ukuran kecil (2-3 variabel) |

Hasil kedua metode harus identik (selisih hanya galat pembulatan).

In [ ]:
def cramer(A, b):
    detA = np.linalg.det(A)             # determinan matriks utama
    if abs(detA) < 1e-12:
        raise ValueError("det(A) = 0, Cramer tidak bisa dipakai.")
    hasil = []
    for i in range(len(b)):
        Ai = A.copy()
        Ai[:, i] = b                    # ganti kolom ke-i dengan b
        hasil.append(np.linalg.det(Ai) / detA)
    return np.array(hasil)

print("Cramer :", cramer(A, b))

## Cell 7 - Uji SPL lain
**Tujuan:** menguji bahwa program berlaku umum, bukan hanya untuk satu contoh.

SPL uji:
```
 x +  y +  z =  6
      2y + 5z = -4
2x + 5y -  z = 27
```
Solusi yang diharapkan: x = 5, y = 3, z = -2. Perhatikan koefisien x di persamaan 2 dan koefisien y di persamaan 1 yang ditulis sesuai posisinya, termasuk 0.

**Latihan mandiri:** ganti `A2` dan `b2` dengan soal buatan Anda sendiri. Coba juga SPL singular, misalnya baris 2 = 2 × baris 1, untuk melihat pesan error.

In [ ]:
A2 = np.array([[1., 1., 1.], [0., 2., 5.], [2., 5., -1.]])
b2 = np.array([6., -4., 27.])
print("Solusi:", gauss_elimination(A2, b2, tampil=False))

## Kesimpulan
- Eliminasi Gauss dengan pivoting parsial berhasil menyelesaikan SPL 3 variabel, dan hasilnya sama dengan Cramer dan NumPy.
- Pivoting parsial menjaga kestabilan numerik dan mencegah pembagian dengan nol.
- Verifikasi dengan residual penting untuk memastikan kebenaran hasil.
- Keterbatasan: untuk SPL sangat besar atau sangat sensitif (*ill-conditioned*), diperlukan metode lain, misalnya dekomposisi LU atau metode iteratif (Jacobi, Gauss-Seidel).

## Ringkasan masalah yang mungkin muncul
| Gejala | Penyebab | Solusi |
|---|---|---|
| `Matriks singular` | det(A) = 0 | Periksa kembali persamaan dan koefisien |
| Hasil tidak cocok dengan NumPy | Salah input / salah indeks | Cetak `A` dan `b`, cocokkan dengan soal |
| Residual besar (> 1e-8) | Matriks *ill-conditioned* | Cek `np.linalg.cond(A)`, pertimbangkan metode lain |
| Error dimensi | Ukuran `A` dan `b` tidak cocok | Pastikan A 3x3 dan b berisi 3 elemen |